# CAGCo-Net on Hateful Memes — final project Nithish (A1953847)
Runtime → Change runtime type → **T4 GPU**. Then run the cells top to bottom.

## 1. Get the code
Clones the project from GitHub: https://github.com/jnithish1/final-project-Nithish-A1953847

In [ ]:
!git clone https://github.com/jnithish1/final-project-Nithish-A1953847.git
%cd final-project-Nithish-A1953847
!ls src

In [ ]:
!pip -q install -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Get the original Hateful Memes dataset
Downloads the original Meta AI release (10,000 PNG memes + jsonl files, about 3.4 GB) from the Hugging Face mirror `neuralcatcher/hateful_memes`. Read `data/hateful_memes/LICENSE.txt` — the data is for research use under Meta's licence.

In [ ]:
# Sign in to Hugging Face first (free account → Settings → Access Tokens → Read token).
# Without signing in, Hugging Face rate-limits the 10,000-image download.
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
%%bash
hf auth whoami
export HF_HUB_DISABLE_XET=1
for i in $(seq 1 20); do
  python -c "
from huggingface_hub import snapshot_download
snapshot_download(repo_id='neuralcatcher/hateful_memes', repo_type='dataset',
                  local_dir='data/hateful_memes', max_workers=4)
" && break
  echo "Rate limited - waiting 60 s, then resuming (attempt $i)"
  sleep 60
done
echo "Images downloaded:"; ls data/hateful_memes/img | wc -l

## 3. Self-check on random data
Send the output of these two cells to Claude if anything fails.

In [ ]:
!python -m src.smoke_test              # CLIP-feature models, ~1 min
!python -m src.smoke_test --finetuned  # ResNet-152, BERT, Faster R-CNN, VisualBERT (downloads weights)

## 4. Full pipeline
Use `--quick` first for a short dry run. The full run takes a few hours on a T4 because ResNet-152, BERT and VisualBERT are fine-tuned for 5 seeds each.

In [ ]:
# Keep results on Drive so a disconnect loses nothing; re-running skips finished runs.
# from google.colab import drive; drive.mount('/content/drive')
# OUT = '/content/drive/MyDrive/cagco_outputs'
OUT = 'outputs'
!python run_all.py --data-dir data/hateful_memes --out-dir {OUT}

## 5. Look at the results

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open(f'{OUT}/RESULTS.md').read()))
import glob
for f in sorted(glob.glob(f'{OUT}/figures/*.png')):
    print(f); display(Image(f, width=700))

In [ ]:
# Save everything back to Drive
# !zip -qr /content/drive/MyDrive/cagconet_outputs.zip outputs -x 'outputs/features/*'